### Building a RAG System with LangChain and ChromaDB
#### Introduction
Retrieval-Augmented Generation (RAG) is a powerful technique that combines the capabilities of large language models with external knowledge retrieval. This notebook will walk you through building a complete RAG system using:

- LangChain: A framework for developing applications powered by language models
- ChromaDB: An open-source vector database for storing and retrieving embeddings
- OpenAI: For embeddings and language model (you can substitute with other providers)

In [73]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [74]:
## langchain imports
from langchain_text_splitters  import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings
#from langchain.schema import Document
from langchain_core.documents import Document


## vector stores
from langchain_community.vectorstores import Chroma

#utility imports
import numpy as np
from typing import List


In [75]:
# RAG Architecture Overview
print("""
RAG (Retrieval-Augmented Generation) Architecture:

1. Document Loading: Load documents from various sources
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query to embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

Benefits of RAG:
- Reduces hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge
""")


RAG (Retrieval-Augmented Generation) Architecture:

1. Document Loading: Load documents from various sources
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query to embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

Benefits of RAG:
- Reduces hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge



### 1. Sample Data

In [76]:
## create sample documents
sample_docs = [
    """
    Machine Learning Fundamentals
    
    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main 
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervised 
    learning finds patterns in unlabeled data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties.
    """,
    
    """
    Deep Learning and Neural Networks
    
    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of interconnected 
    nodes. Deep learning has revolutionized fields like computer vision, natural language 
    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly 
    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers 
    excel at sequential data processing.
    """,
    
    """
    Natural Language Processing (NLP)
    
    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, 
    machine translation, and question answering. Modern NLP heavily relies on transformer 
    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand 
    context and relationships between words in text.
    """
]

sample_docs


['\n    Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.\n    ',
 '\n    Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective f

In [77]:
## save sample documents to files
import tempfile
temp_dir=tempfile.mkdtemp()

for i,doc in enumerate(sample_docs):
    with open(f"doc_{i}.txt","w") as f:
        f.write(doc)

In [78]:
temp_dir

'C:\\Users\\SAMARJ~1\\AppData\\Local\\Temp\\tmp9s5y_o1e'

### 2. Document Loading

In [79]:
from langchain_community.document_loaders import DirectoryLoader,TextLoader

# Load documents from directory
loader = DirectoryLoader(
    "data", 
    glob="*.txt", 
    loader_cls=TextLoader,
    loader_kwargs={'encoding': 'utf-8'}
)
documents = loader.load()

print(f"Loaded {len(documents)} documents")
print(f"\nFirst document preview:")
print(documents[0].page_content[:200] + "...")


Loaded 3 documents

First document preview:

    Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. Ther...


### Document Splitting

In [80]:
# Initialize text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,  # Maximum size of each chunk
    chunk_overlap=50,  # Overlap between chunks to maintain context
    length_function=len,
    separators=[" "]  # Hierarchy of separators
)
chunks=text_splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks from {len(documents)} documents")
print(f"\nChunk example:")
print(f"Content: {chunks[0].page_content[:150]}...")
print(f"Metadata: {chunks[0].metadata}")

Created 5 chunks from 3 documents

Chunk example:
Content: Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experie...
Metadata: {'source': 'data\\doc_0.txt'}


In [81]:
chunks


[Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through'),
 Document(metadata={'source': 'data\\doc_0.txt'}, page_content='data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.'),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of in

### Embedding Models

In [82]:
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

In [83]:
sample_text="MAchine LEarning is fascinating"
embeddings=OpenAIEmbeddings()
embeddings

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x0000025B91EA7B10>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x0000025B91EA7D90>, model='text-embedding-ada-002', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [84]:
vector=embeddings.embed_query(sample_text)
vector

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

### Intilialize the ChromaDB Vector Store And Stores the chunks in Vector Representation

In [ ]:
chunks


[Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through'),
 Document(metadata={'source': 'data\\doc_0.txt'}, page_content='data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.'),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of in

In [ ]:
## Create a Chromdb vector store
persist_directory="./chroma_db"

## Initialize Chromadb with Open AI embeddings
vectorstore=Chroma.from_documents(
    documents=chunks,
    embedding=OpenAIEmbeddings(),
    persist_directory=persist_directory,
    collection_name="rag_collection"

)

print(f"Vector store created with {vectorstore._collection.count()} vectors")
print(f"Persisted to: {persist_directory}")

Vector store created with 5 vectors
Persisted to: ./chroma_db


### Test Similarity Search

In [ ]:
query="What are the types of machine learning?"

similar_docs=vectorstore.similarity_search(query,k=3)
similar_docs

[Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through'),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (

In [ ]:
query="what is NLP?"

similar_docs=vectorstore.similarity_search(query,k=3)
similar_docs

[Document(metadata={'source': 'data\\doc_2.txt'}, page_content='Natural Language Processing (NLP)\n\n    NLP is a field of AI that focuses on the interaction between computers and human language. \n    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, \n    machine translation, and question answering. Modern NLP heavily relies on transformer \n    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand \n    context and relationships between words in text.'),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are part

In [ ]:
print(f"Query: {query}")
print(f"\nTop {len(similar_docs)} similar chunks:")
for i, doc in enumerate(similar_docs):
    print(f"\n--- Chunk {i+1} ---")
    print(doc.page_content[:200] + "...")
    print(f"Source: {doc.metadata.get('source', 'Unknown')}")

Query: what is NLP?

Top 3 similar chunks:

--- Chunk 1 ---
Natural Language Processing (NLP)

    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity recogn...
Source: data\doc_2.txt

--- Chunk 2 ---
Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of i...
Source: data\doc_1.txt

--- Chunk 3 ---
Neural Networks (RNNs) and Transformers 
    excel at sequential data processing....
Source: data\doc_1.txt


### Advanced Similarity Search With Scores

In [ ]:
results_scores=vectorstore.similarity_search_with_score(query,k=3)
results_scores

[(Document(metadata={'source': 'data\\doc_2.txt'}, page_content='Natural Language Processing (NLP)\n\n    NLP is a field of AI that focuses on the interaction between computers and human language. \n    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, \n    machine translation, and question answering. Modern NLP heavily relies on transformer \n    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand \n    context and relationships between words in text.'),
  0.23339326679706573),
 (Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neura

#### Understanding Similarity Scores
The similarity score represents how closely related a document chunk is to your query. The scoring depends on the distance metric used:

ChromaDB default: Uses L2 distance (Euclidean distance)

- Lower scores = MORE similar (closer in vector space)
- Score of 0 = identical vectors
- Typical range: 0 to 2 (but can be higher)


Cosine similarity (if configured):

- Higher scores = MORE similar
- Range: -1 to 1 (1 being identical)

### Standard RAG System Implementation

Now we'll build a Retrieval-Augmented Generation (RAG) system using the most common and widely adopted pattern in the LangChain community. This approach uses Runnables to create a clean, modular RAG pipeline.

In [ ]:
from langchain_openai import ChatOpenAI

llm=ChatOpenAI(
    model_name="gpt-3.5-turbo"
)


In [ ]:
test_response=llm.invoke("What is Large Language Models")
test_response

AIMessage(content="Large Language Models (LLMs) are a type of artificial intelligence that uses deep learning techniques to understand and generate human language. These models are trained on large amounts of text data, such as books, articles, and other written content, to learn the patterns and relationships within language. LLMs are able to generate human-like text, answer questions, and perform a variety of other language-related tasks. Examples of popular LLMs include OpenAI's GPT (Generative Pre-trained Transformer) models and Google's BERT (Bidirectional Encoder Representations from Transformers).", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 114, 'prompt_tokens': 12, 'total_tokens': 126, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name'

In [ ]:
from langchain.chat_models.base import init_chat_model

llm=init_chat_model("openai:gpt-3.5-turbo")
#llm=init_chat_model("groq:")
llm

ChatOpenAI(output_version=None, profile={'name': 'GPT-3.5-turbo', 'release_date': '2023-03-01', 'last_updated': '2023-11-06', 'open_weights': False, 'max_input_tokens': 16385, 'max_output_tokens': 4096, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': False, 'structured_output': False, 'attachment': False, 'temperature': True, 'image_url_inputs': False, 'pdf_inputs': False, 'pdf_tool_message': False, 'image_tool_message': False, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x0000023B6FB58A50>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000023B6FB58CD0>, root_client=<openai.OpenAI object at 0x0000023B6FA7B490>, root_async_client=<openai.AsyncOpenAI object at 0x0000023B6FAC82B0>, model_kwargs={}, openai_api_key=Sec

In [ ]:
llm.invoke("What is AI")

AIMessage(content='AI, or artificial intelligence, refers to the simulation of human intelligence processes by machines, typically computer systems. This includes learning, problem solving, and decision making. AI technology enables machines to perform tasks that would typically require human intelligence, such as recognizing speech, making predictions, and understanding natural language. AI has a wide range of applications across various industries, including healthcare, finance, transportation, and more.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 80, 'prompt_tokens': 10, 'total_tokens': 90, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DVNDtRWhtIZjZD8N1JcrERQTfKriS', 's

### Modern RAG Chain | LCEL (langchain expression language)

In [ ]:
# Standard RAG System - Most Common Implementation Pattern
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

print("✓ Imports successful!")

# 1. Create a standard RAG prompt template
rag_prompt = ChatPromptTemplate.from_template("""
You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.

Context: {context}

Question: {question}

Answer:
""")

print("✓ RAG prompt template created!")

# 2. Create retriever from vectorstore  
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("✓ Retriever created!")

# 3. Function to format retrieved documents
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 4. Build the standard RAG chain using Runnables (Most common pattern)
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("✓ Standard RAG chain built successfully!")
print("This follows the most widely adopted RAG pattern in the LangChain community.")

✓ Imports successful!
✓ RAG prompt template created!
✓ Retriever created!
✓ Standard RAG chain built successfully!
This follows the most widely adopted RAG pattern in the LangChain community.


In [ ]:
# Test the Standard RAG System
test_questions = [
    "What are the types of machine learning?",
    "What is deep learning?", 
    "Explain NLP and its applications",
    "What are CNNs used for?"
]

print("🚀 Testing Standard RAG System")
print("=" * 50)

for i, question in enumerate(test_questions, 1):
    print(f"\n{i}. Question: {question}")
    response = rag_chain.invoke(question)
    print(f"   Answer: {response}")
    print("-" * 30)

🚀 Testing Standard RAG System

1. Question: What are the types of machine learning?
   Answer: The types of machine learning are supervised learning, unsupervised learning, and reinforcement learning. Supervised learning uses labeled data, unsupervised learning finds patterns in unlabeled data, and reinforcement learning learns through interactions with an environment.
------------------------------

2. Question: What is deep learning?
   Answer: Deep learning is a subset of machine learning that relies on artificial neural networks inspired by the human brain with interconnected layers of nodes. It has revolutionized fields like computer vision, natural language processing, and speech recognition. Deep learning encompasses models like Convolutional Neural Networks (CNNs) for image processing, Recurrent Neural Networks (RNNs), and Transformers.
------------------------------

3. Question: Explain NLP and its applications
   Answer: NLP is a field of AI focused on computer-human languag

In [ ]:
# Simple query function for the RAG system
def ask_rag(question: str) -> str:
    """
    Ask a question to the RAG system
    
    Args:
        question (str): The question to ask
        
    Returns:
        str: The RAG system's response
    """
    try:
        response = rag_chain.invoke(question)
        return response
    except Exception as e:
        return f"Error: {str(e)}"

# Example usage
print("🎯 RAG System Ready!")
print("\nExample usage:")
print("• ask_rag('What is reinforcement learning?')")
print("• ask_rag('How do CNNs work?')")
print("• ask_rag('What are transformer models?')")

# Test the function
print(f"\n📝 Test Query: 'What is reinforcement learning?'")
print(f"🤖 Answer: {ask_rag('What is reinforcement learning?')}")

print(f"\n📝 Test Query: 'How do transformer models work?'")  
print(f"🤖 Answer: {ask_rag('How do transformer models work?')}")

🎯 RAG System Ready!

Example usage:
• ask_rag('What is reinforcement learning?')
• ask_rag('How do CNNs work?')
• ask_rag('What are transformer models?')

📝 Test Query: 'What is reinforcement learning?'
🤖 Answer: Reinforcement learning is a type of machine learning that learns through interaction with an environment using rewards and penalties. It is one of the three main types of machine learning, alongside supervised and unsupervised learning. Unlike supervised learning, reinforcement learning does not use labeled data to train models.

📝 Test Query: 'How do transformer models work?'
🤖 Answer: Transformer models work by using attention mechanisms to understand context and relationships between words in text. They consist of layers that process input data in parallel, allowing for more efficient training. Transformers have revolutionized fields like Natural Language Processing by achieving state-of-the-art performance in tasks such as machine translation and question answering.


In [ ]:
# Comprehensive RAG System Demo
def demo_rag_system():
    """
    Comprehensive demonstration of the RAG system with retrieval context
    """
    print("🚀 RAG SYSTEM COMPREHENSIVE DEMO")
    print("=" * 60)
    
    test_questions = [
        "What are the three types of machine learning?",
        "What is deep learning and how does it relate to neural networks?", 
        "What are CNNs best used for?",
        "Explain the key tasks in NLP"
    ]
    
    for i, question in enumerate(test_questions, 1):
        print(f"\n{i}. Question: {question}")
        print("-" * 50)
        
        # Get the answer
        answer = ask_rag(question)
        print(f"Answer: {answer}")
        
        # Show retrieved context
        retrieved_docs = retriever.invoke(question)
        print("\n📄 Retrieved Context Sources:")
        for j, doc in enumerate(retrieved_docs):
            print(f"  Source {j+1}: {doc.page_content[:100]}...")
            if doc.metadata:
                print(f"    Metadata: {doc.metadata}")
        
        print("\n" + "="*60)

# Run the comprehensive demo
demo_rag_system()

print("\n✅ RAG System Successfully Implemented!")
print("📚 This follows the most common and standard RAG pattern used in the community.")
print("\n🎯 Usage Examples:")
print("• ask_rag('Your question here')")
print("• rag_chain.invoke('Your question here')")

🚀 RAG SYSTEM COMPREHENSIVE DEMO

1. Question: What are the three types of machine learning?
--------------------------------------------------
Answer: The three types of machine learning are supervised learning, unsupervised learning, and reinforcement learning. Supervised learning uses labeled data, while unsupervised learning finds patterns in unlabeled data. Reinforcement learning learns through trial and error.

📄 Retrieved Context Sources:
  Source 1: Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enab...
    Metadata: {'source': 'data\\doc_0.txt'}
  Source 2: Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artifi...
    Metadata: {'source': 'data\\doc_1.txt'}
  Source 3: Natural Language Processing (NLP)

    NLP is a field of AI that focuses on the interaction between ...
    Metadata: {'source': 'data\\doc_2.txt'}


2. Question: What is deep learning and how does it relate to neural n

### Advanced Rag Techniques- Conversational Memory
Understanding Conversational Memory in RAG
Conversational memory enables RAG systems to maintain context across multiple interactions. This is crucial for:

Follow-up questions that reference previous answers
Pronoun resolution (e.g., "it", "they", "that")
Context-dependent queries that build on prior discussion
Natural dialogue flow where users don't repeat context

Key Challenge:
Traditional RAG retrieves documents based only on the current query, missing important context from the conversation. For example:

User: "Tell me about Python"
Bot: explains Python programming language
User: "What are its main libraries?" ← "its" refers to Python, but retriever doesn't know this

Solution:
The modern approach uses a two-step process:

Query Reformulation: Transform context-dependent questions into standalone queries
Context-Aware Retrieval: Use the reformulated query to fetch relevant documents

In [ ]:
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

### Updated Conversational RAG with RunnableWithMessageHistory

Now let's implement the same conversational flow but using the modern `RunnableWithMessageHistory` approach instead of the problematic `create_history_aware_retriever`.

In [ ]:
# Updated implementation using RunnableWithMessageHistory (Modern Approach)
# This replaces the old create_history_aware_retriever pattern

# Create session-based chat history management
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import BaseMessage

class InMemoryChatMessageHistory(BaseChatMessageHistory):
    """In-memory implementation of chat message history"""
    
    def __init__(self):
        self.messages: list[BaseMessage] = []
    
    def add_message(self, message: BaseMessage) -> None:
        """Add a message to the store"""
        self.messages.append(message)
    
    def clear(self) -> None:
        """Clear all messages from the store"""
        self.messages = []

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    """Return a chat message history for the given session_id"""
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# Initialize session store (if not already created)
if 'store' not in locals():
    store = {}

# Create the conversational RAG chain for history-aware context
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

# Create context-aware prompt that handles history
contextualize_q_system_prompt = """Given a chat history and the latest user question 
which might reference context in the chat history, formulate a standalone question 
which can be understood without the chat history. Do NOT answer the question, 
just reformulate it if needed and otherwise return it as is."""

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

# Create the context-aware question reformulation chain
contextualize_q_chain = contextualize_q_prompt | llm | StrOutputParser()

# Create the context-retrieval function with history awareness
def get_contextualized_retrieval(input_dict):
    if input_dict.get("chat_history"):
        # If there's history, reformulate the question for better retrieval
        reformulated_question = contextualize_q_chain.invoke(input_dict)
        return retriever.invoke(reformulated_question)
    else:
        # If no history, use the original question
        return retriever.invoke(input_dict["input"])

# Create the QA prompt with history support
qa_system_prompt = """You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.

Context: {context}"""

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", qa_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

# Create the conversational RAG chain
conversational_rag_chain = (
    RunnablePassthrough.assign(
        context=get_contextualized_retrieval
    )
    | qa_prompt
    | llm
)

# Wrap with RunnableWithMessageHistory for automatic session management
conversational_rag_with_history = RunnableWithMessageHistory(
    conversational_rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)

print("✅ Modern Conversational RAG chain created using RunnableWithMessageHistory!")
print("✅ Ready to handle multi-turn conversations with automatic history management")

✅ Modern Conversational RAG chain created using RunnableWithMessageHistory!
✅ Ready to handle multi-turn conversations with automatic history management


In [85]:
# Demonstration: Same conversational flow as original but with modern approach
print("🚀 Testing Updated Conversational RAG with RunnableWithMessageHistory")
print("=" * 70)

# Create a new session for this demo
demo_session = "demo_session_updated"
config = {"configurable": {"session_id": demo_session}}

# First question (equivalent to your original first question)
print("🗣️ First Question:")
result1 = conversational_rag_with_history.invoke({
    "input": "What is machine learning?"
}, config=config)

print(f"Q: What is machine learning?")
print(f"A: {result1.content}")
print("\n" + "-" * 50)

# Follow-up question (equivalent to your original follow-up)
print("🗣️ Follow-up Question (references previous context):")
result2 = conversational_rag_with_history.invoke({
    "input": "What are its main types?"  # This "its" refers to ML from previous question
}, config=config)

print(f"Q: What are its main types?")
print(f"A: {result2.content}")
print("\n" + "-" * 50)

# Additional follow-up to show continuing context
print("🗣️ Another Follow-up Question:")
result3 = conversational_rag_with_history.invoke({
    "input": "Tell me more about the first type you mentioned"
}, config=config)

print(f"Q: Tell me more about the first type you mentioned")
print(f"A: {result3.content}")

print("\n" + "=" * 70)
print("✅ SUCCESS: Modern approach working perfectly!")
print("✅ No manual chat_history management needed")
print("✅ Automatic session-based conversation tracking")
print("✅ Same conversational flow as original code but more robust")

# Show the conversation history
print(f"\n📋 Session '{demo_session}' Conversation History:")
session_history = store[demo_session]
for i, msg in enumerate(session_history.messages):
    role = "🧑 Human" if isinstance(msg, HumanMessage) else "🤖 AI"
    print(f"{i+1}. {role}: {msg.content[:80]}...")

print(f"\n📊 Total messages in session: {len(session_history.messages)}")

🚀 Testing Updated Conversational RAG with RunnableWithMessageHistory
🗣️ First Question:


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}